# HW3,  : direct and iterative solvers

We solve the Friedkin-Johnsen equilibrium system

$$(I - \alpha W)\,x = (1 - \alpha)\,s, \qquad A = I - \alpha W, \quad b = (1 - \alpha)\,s$$

on the soc-Epinions1 trust network, with $\alpha = 0.85$.

All the code lives in `hw3_system.py` (system construction) and `solvers.py` (Jacobi, Gauss-Seidel, stopping criteria). This notebook only calls those functions and comments the results. Run it from the repository root.

## 1. System construction and reference solution

**Goal.** Rebuild the matrix $A$ of Member 1 (`01_dataset_preprocessing.ipynb`), check that it matches her numbers, define the right-hand side $b$ and compute a reference solution $x^*$ to validate the iterative solvers against.

**Method.**
- `build_system(alpha)` reads `data/processed/edges_indexed.csv` and repeats Member 1's logic: $W_{ij} = 1/d^{out}_i$ for every edge $i \to j$ (sink rows stay zero), then $A = I - \alpha W$ in CSR format.
- `rcm_permutation(A)` is Member 1's Reverse Cuthill-McKee reordering.
- We verify numerically the properties that guarantee convergence of Jacobi and Gauss-Seidel: unit diagonal, strict diagonal dominance by rows, $\|\alpha W\|_\infty = \alpha$.
- $s_i \sim \mathcal{U}[-1, 1]$ with seed 42 (agreed with the group), saved to `data/processed/m2_s_initial_opinions.npy` so everybody uses the same vector.
- $x^*$ is computed once with the sparse LU factorization `scipy.sparse.linalg.splu` and saved to disk.

In [1]:
import time
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import splu

from hw3_system import build_system, bandwidth, rcm_permutation, initial_opinions, processed_folder

alpha = 0.85

In [2]:
# 1.1 Build W and A = I - alpha W

start = time.perf_counter()
A, W = build_system(alpha)
print(f"build time: {time.perf_counter() - start:.2f} s")

n = A.shape[0]
csr_mb = (A.data.nbytes + A.indices.nbytes + A.indptr.nbytes) / 1024**2

print("shape:", A.shape)
print("nnz(W):", W.nnz, "  nnz(A):", A.nnz, "  (= nnz(W) + n:", W.nnz + n == A.nnz, ")")
print(f"CSR memory of A: {csr_mb:.2f} MB")

build time: 0.03 s
shape: (75879, 75879)
nnz(W): 508837   nnz(A): 584716   (= nnz(W) + n: True )
CSR memory of A: 6.98 MB


In [3]:
# 1.2 Bandwidth before and after RCM, and save the permutation
# RCM breaks ties between nodes of equal degree arbitrarily, so the exact
# permutation can differ between machines. We save ours for reproducibility.

p_path = processed_folder / "m2_rcm_permutation.npy"
if p_path.exists():
    p = np.load(p_path)
else:
    p = rcm_permutation(A)
    np.save(p_path, p)

A_rcm = A[p, :][:, p]
print("Bw(A):", bandwidth(A))
print("Bw(A) after RCM:", bandwidth(A_rcm))

# how many nodes share the same degree in the pattern of A + A^T (RCM ties)
degree = np.diff((A + A.T).tocsr().indptr)
print("nodes with the most common degree:", np.bincount(degree).max(), "of", n)

Bw(A): 75407
Bw(A) after RCM: 44499
nodes with the most common degree: 35755 of 75879


In [4]:
# 1.3 Sanity checks on W and A

row_sums_W = np.asarray(W.sum(axis=1)).ravel()
sink = row_sums_W == 0
print("diag(W) all zero (no self-loops):", np.all(W.diagonal() == 0))
print("row sums of W are 1 or 0:", np.allclose(row_sums_W[~sink], 1.0), "| sink rows:", sink.sum())

diag_A = A.diagonal()
offdiag_abs_sum = np.asarray(abs(A).sum(axis=1)).ravel() - np.abs(diag_A)
print("diag(A) = 1, so D = I:", np.allclose(diag_A, 1.0))
print("max off-diagonal row sum of |A|:", offdiag_abs_sum.max())
print("strictly diagonally dominant by rows:", np.all(np.abs(diag_A) > offdiag_abs_sum))

# Jacobi iteration matrix B_J = D^{-1}(D - A) = alpha W, infinity norm = max row sum
norm_BJ = np.asarray(abs(alpha * W).sum(axis=1)).ravel().max()
print("||B_J||_inf = ||alpha W||_inf:", norm_BJ)
print("A symmetric:", (A - A.T).nnz == 0)
print(f"bound K_inf(A) <= (1 + alpha) / (1 - alpha) = {(1 + alpha) / (1 - alpha):.2f}")

diag(W) all zero (no self-loops): True
row sums of W are 1 or 0: True | sink rows: 15538
diag(A) = 1, so D = I: True
max off-diagonal row sum of |A|: 0.8500000000000016
strictly diagonally dominant by rows: True
||B_J||_inf = ||alpha W||_inf: 0.8500000000000004
A symmetric: False
bound K_inf(A) <= (1 + alpha) / (1 - alpha) = 12.33


In [5]:
# 1.4 Right-hand side b = (1 - alpha) s

s = initial_opinions(n)
b = (1 - alpha) * s
print(f"s: min {s.min():.4f}, max {s.max():.4f}, mean {s.mean():.4f}, std {s.std():.4f}")
print("first 5 entries of s:", np.round(s[:5], 4))

s: min -1.0000, max 1.0000, mean 0.0017, std 0.5768
first 5 entries of s: [ 0.5479 -0.1222  0.7172  0.3947 -0.8116]


In [6]:
# 1.5 Reference solution x* with sparse LU (computed once, then loaded)
# splu wants CSC. permc_spec="COLAMD" reorders the columns to limit fill-in.

x_ref_path = processed_folder / "m2_x_reference_alpha085.npy"
if x_ref_path.exists():
    x_ref = np.load(x_ref_path)
    print("x* loaded from", x_ref_path)
else:
    start = time.perf_counter()
    lu = splu(A.tocsc(), permc_spec="COLAMD")
    lu_time = time.perf_counter() - start
    x_ref = lu.solve(b)
    np.save(x_ref_path, x_ref)

    lu_nnz = lu.L.nnz + lu.U.nnz
    print(f"splu time: {lu_time:.1f} s")
    print(f"nnz(L) + nnz(U): {lu_nnz:,}  fill ratio: {lu_nnz / A.nnz:.1f}x nnz(A)")
    print(f"approx. memory of L and U: {lu_nnz * 12 / 1024**2:,.0f} MB (8 B value + 4 B index per nonzero)")
    del lu

rel_res = np.linalg.norm(b - A @ x_ref, np.inf) / np.linalg.norm(b, np.inf)
print(f"relative residual ||b - A x*||_inf / ||b||_inf = {rel_res:.2e}")

x* loaded from data/processed/m2_x_reference_alpha085.npy
relative residual ||b - A x*||_inf / ||b||_inf = 2.39e-14


In [7]:
# 1.6 Checks on x*: range, and sink rows (row i of A is e_i^T, so x_i = (1 - alpha) s_i)

print(f"x*: min {x_ref.min():.4f}, max {x_ref.max():.4f}, std {x_ref.std():.4f} (std of s: {s.std():.4f})")
print("x* inside [-1, 1]:", np.all(np.abs(x_ref) <= 1))
print("sink nodes: x*_i = (1 - alpha) s_i:", np.allclose(x_ref[sink], (1 - alpha) * s[sink]))

x*: min -0.9516, max 0.8922, std 0.1182 (std of s: 0.5768)
x* inside [-1, 1]: True
sink nodes: x*_i = (1 - alpha) s_i: True


### Results of Section 1

**The system matches Member 1.** $A$ has shape $75{,}879 \times 75{,}879$, $\mathrm{nnz}(A) = 584{,}716 = \mathrm{nnz}(W) + n$ (every diagonal entry is stored, sink rows included) and $Bw(A) = 75{,}407$, as in `01_dataset_preprocessing.ipynb`. In CSR it takes about 7 MB.

**RCM bandwidth.** We obtain $Bw = 44{,}499$ after RCM, while Member 1 reported $45{,}409$. The difference is not an error. In the pattern of $A + A^T$, 35,755 nodes share the same degree, and RCM breaks these ties in an arbitrary order that depends on the sorting routine of the machine. Relabelling the nodes at random before RCM gives bandwidths between about 44,200 and 45,000. In both cases RCM only reduces the bandwidth by about 40%: the hubs of a scale-free graph are connected to nodes all over the matrix, so no ordering can make $A$ narrow-banded. Our permutation is saved in `data/processed/m2_rcm_permutation.npy` so that the Gauss-Seidel experiments (which depend on the ordering) are reproducible.

**Convergence is guaranteed.** The checks confirm the theory:
- $W$ has zero diagonal and row sums equal to 1 (or 0 for the 15,538 sinks), so $\mathrm{diag}(A) = 1$ and $D = I$.
- The off-diagonal row sums of $|A|$ are at most $\alpha = 0.85 < 1 = |a_{ii}|$, so $A$ is strictly diagonally dominant by rows. Jacobi and Gauss-Seidel therefore both converge for any $x^{(0)}$.
- The Jacobi iteration matrix is $B_J = I - A = \alpha W$ with $\|B_J\|_\infty = 0.85$, hence $\rho(B_J) \le 0.85$. The error contracts at least by a factor 0.85 per iteration, so at most about $\log(10^{-8}) / \log(0.85) \approx 113$ iterations are needed for a $10^{-8}$ reduction.
- $A$ is not symmetric, so the SPD convergence theorem for Gauss-Seidel does not apply. Diagonal dominance is the argument we use.
- $K_\infty(A) \le (1 + \alpha)/(1 - \alpha) \approx 12.3$: the system is well conditioned at $\alpha = 0.85$.

**Reference solution.** The sparse LU factorization (`splu` with COLAMD column ordering) gives a relative residual of $2.4 \times 10^{-14}$. With $K_\infty(A) \le 12.3$ the relative error of $x^*$ is at most about $3 \times 10^{-13}$, so $x^*$ is accurate enough to measure the true error of the iterative solvers down to tolerance $10^{-10}$. The factorization is expensive, though: about 150 s and $\mathrm{nnz}(L) + \mathrm{nnz}(U) \approx 1.5 \times 10^8$, i.e. 261 times $\mathrm{nnz}(A)$ (about 1.7 GB instead of 7 MB). This fill-in is studied in Section 5. $x^*$ is saved in `data/processed/m2_x_reference_alpha085.npy`.

**Check of the model.** All equilibrium opinions lie in $[-1, 1]$, as expected since each $x_i$ is a weighted average of the $s_j$ with nonnegative weights summing to at most 1. For sink nodes $x^*_i = (1 - \alpha) s_i = 0.15\, s_i$ exactly. A zero row of $W$ therefore does not mean that a sink "retains 100% of its initial opinion" (as stated in the preprocessing notebook): its opinion is scaled toward 0. If the group wants sinks to keep $s_i$, a self-loop $W_{ii} = 1$ for sinks or $b_i = s_i$ on sink rows would do it. This is a modelling choice for the group and does not affect the convergence analysis, since diagonal dominance holds in all three variants.

## 2. Jacobi method

**Goal.** Implement the Jacobi method from scratch in two ways (element by element on the CSR arrays, and in matrix form), validate it on a small system with known solution and on the full system against $x^*$, and compare the cost per iteration of the two versions.

**Method.** Split $A = D - (D - A)$ with $D = \mathrm{diag}(A)$. Then $Ax = b$ becomes $Dx = b - (A - D)x$, which suggests the iteration

$$x^{(k+1)} = D^{-1}\left(b - (A - D)\,x^{(k)}\right), \qquad B_J = D^{-1}(D - A).$$

- **Loop version** (`mode="loop"`): for every row $i$, $x^{(k+1)}_i = \big(b_i - \sum_{j \ne i} a_{ij} x^{(k)}_j\big) / a_{ii}$, reading the nonzeros of row $i$ from `indptr`, `indices`, `data`. Only the old vector $x^{(k)}$ is used, so the rows could be updated in any order (or in parallel).
- **Matrix version** (`mode="matrix"`): one sparse matrix-vector product with the off-diagonal part $A - D$ (stored once in CSR) and one element-wise division by the diagonal.

For our system $D = I$ and $B_J = \alpha W$, so Jacobi reads $x^{(k+1)} = b + \alpha W x^{(k)}$: at every step each user takes their own (scaled) opinion plus $\alpha$ times the average opinion of the people they trust. The solver nevertheless divides by the actual diagonal, so it works for any matrix with nonzero diagonal.

Both versions share the same stopping logic (`_iterate` in `solvers.py`), which logs the three criteria of Section 4 at every iteration. Here we stop on the relative residual $\|b - Ax^{(k)}\|_\infty / \|b\|_\infty \le 10^{-8}$, with $x^{(0)} = 0$.

In [8]:
# 2.1 Validation on a small strictly diagonally dominant matrix with known solution
# As in the labs: choose x_true = ones and set b = A @ x_true.

from solvers import jacobi, _jacobi_sweep

rng = np.random.default_rng(42)
M = rng.uniform(-1, 1, size=(6, 6))
np.fill_diagonal(M, np.abs(M).sum(axis=1) + 1)  # |m_ii| > sum_{j != i} |m_ij|
A_small = sp.csr_matrix(M)
b_small = A_small @ np.ones(6)

for mode in ["matrix", "loop"]:
    x_small, hist_small = jacobi(A_small, b_small, tol=1e-12, mode=mode)
    print(f"{mode:6s}: {hist_small.attrs['reason']}, max |x - 1| = {np.abs(x_small - 1).max():.1e}")

matrix: residual <= 1e-12 at iteration 26, max |x - 1| = 8.9e-13
loop  : residual <= 1e-12 at iteration 26, max |x - 1| = 8.9e-13


In [9]:
# 2.2 Full system, matrix version, compared with the reference solution x*

x_jac, hist_jac = jacobi(A, b, tol=1e-8, criterion="residual", mode="matrix")

rel_err = np.linalg.norm(x_jac - x_ref, np.inf) / np.linalg.norm(x_ref, np.inf)
print(hist_jac.attrs)
print(f"relative error ||x - x*||_inf / ||x*||_inf = {rel_err:.2e}")
print(f"expected at most about log(1e-8) / log(0.85) = {np.log(1e-8) / np.log(0.85):.0f} iterations")
print(f"time per iteration: {hist_jac.time_s.mean() * 1e3:.2f} ms, total {hist_jac.time_s.sum():.3f} s")
hist_jac.iloc[[0, 1, 2, 49, 99, -1]]

{'converged': True, 'reason': 'residual <= 1e-08 at iteration 114'}
relative error ||x - x*||_inf / ||x*||_inf = 8.99e-09
expected at most about log(1e-8) / log(0.85) = 113 iterations
time per iteration: 0.47 ms, total 0.054 s


,iter,rel_change,abs_change,rel_residual,time_s
0,1,1.000000e+00,1.499981e-01,8.500000e-01,0.000606
1,2,4.632036e-01,1.274984e-01,7.224758e-01,0.000519
2,3,2.865315e-01,1.083700e-01,6.141045e-01,0.000524
49,50,5.485982e-05,5.219137e-05,2.957548e-04,0.000443
99,100,1.622080e-08,1.543636e-08,8.747381e-08,0.000476
113,114,1.667006e-09,1.586390e-09,8.989654e-09,0.000570


In [10]:
# 2.3 Observed contraction factor: ratio between consecutive residuals

ratio = hist_jac.rel_residual.to_numpy()[1:] / hist_jac.rel_residual.to_numpy()[:-1]
print("residual ratio, first 5 iterations:", np.round(ratio[:5], 5))
print("residual ratio, last 5 iterations: ", np.round(ratio[-5:], 5))

# Why exactly 0.85: closed groups of users who only trust each other.
# On such a group W is row-stochastic, so it has eigenvalue 1 and rho(alpha W) = alpha.
from scipy.sparse.csgraph import connected_components
n_comp, label = connected_components(W, directed=True, connection="strong")
r_idx, c_idx = W.nonzero()
has_exit = np.zeros(n_comp, dtype=bool)
has_exit[label[r_idx][label[r_idx] != label[c_idx]]] = True
comp_size = np.bincount(label)
closed = (~has_exit) & (comp_size > 1)
print("closed groups of users (no trust edge leaving them):", closed.sum(), "| sizes from", comp_size[closed].min(), "to", comp_size[closed].max())

residual ratio, first 5 iterations: [0.84997 0.85    0.85    0.85    0.85   ]
residual ratio, last 5 iterations:  [0.85 0.85 0.85 0.85 0.85]
closed groups of users (no trust edge leaving them): 330 | sizes from 2 to 7


In [11]:
# 2.4 Loop version on the full system: same iterates, much slower

x_loop, hist_loop = jacobi(A, b, tol=1e-8, criterion="residual", mode="loop")

print(hist_loop.attrs)
print(f"max |x_loop - x_matrix| = {np.abs(x_loop - x_jac).max():.1e}")
print(f"time per iteration: loop {hist_loop.time_s.mean() * 1e3:.1f} ms, matrix {hist_jac.time_s.mean() * 1e3:.2f} ms "
      f"(loop is {hist_loop.time_s.mean() / hist_jac.time_s.mean():.0f}x slower)")

{'converged': True, 'reason': 'residual <= 1e-08 at iteration 114'}
max |x_loop - x_matrix| = 2.2e-16
time per iteration: loop 101.8 ms, matrix 0.47 ms (loop is 216x slower)


In [12]:
# 2.5 Optional: the same loop compiled with numba (just-in-time compilation to machine code)

from numba import njit

jacobi_sweep_numba = njit(_jacobi_sweep)
diag = A.diagonal()
x_old = np.zeros(n)
jacobi_sweep_numba(A.indptr, A.indices, A.data, diag, b, x_old, np.empty(n))  # first call compiles

n_rep = 100
start = time.perf_counter()
for _ in range(n_rep):
    x_numba = jacobi_sweep_numba(A.indptr, A.indices, A.data, diag, b, x_old, np.empty(n))
numba_time = (time.perf_counter() - start) / n_rep

print(f"one sweep: numba {numba_time * 1e3:.2f} ms vs matrix {hist_jac.time_s.mean() * 1e3:.2f} ms vs python loop {hist_loop.time_s.mean() * 1e3:.1f} ms")
print("numba sweep from x0 = 0 equals the first Jacobi iterate b / diag:", np.allclose(x_numba, b / diag))

one sweep: numba 0.95 ms vs matrix 0.47 ms vs python loop 101.8 ms
numba sweep from x0 = 0 equals the first Jacobi iterate b / diag: True


### Results of Section 2

**Validation.** On the $6 \times 6$ strictly diagonally dominant test matrix both versions reach the exact solution $x = (1, \dots, 1)$ with error $8.9 \times 10^{-13}$ in 26 iterations. On the full system Jacobi stops after 114 iterations with relative error $\|x - x^*\|_\infty / \|x^*\|_\infty = 9.0 \times 10^{-9}$, consistent with the tolerance $10^{-8}$.

**The convergence speed is exactly the one predicted by the theory.** With $D = I$ one Jacobi step is $x^{(k+1)} = x^{(k)} + r^{(k)}$, where $r^{(k)} = b - Ax^{(k)}$. Substituting gives $r^{(k+1)} = \alpha W\, r^{(k)}$, hence $r^{(k)} = (\alpha W)^k b$ for $x^{(0)} = 0$ and

$$\frac{\|r^{(k)}\|_\infty}{\|b\|_\infty} \le \|\alpha W\|_\infty^k = 0.85^k .$$

The bound requires $k \ge \log(10^{-8}) / \log(0.85) = 113.3$, i.e. 114 iterations, which is exactly what we observe. The residual ratio between consecutive iterations is $0.85$ at every step, so the bound is attained, not just satisfied. The reason is in the graph: there are 330 closed groups of 2 to 7 users who only trust each other. On each of them $W$ is row-stochastic, so $W$ has eigenvalue 1 and $\rho(B_J) = \rho(\alpha W) = \alpha$ exactly. For Jacobi the number of iterations therefore grows like $\log(\mathrm{tol}) / \log(\alpha)$, which becomes large as the dogmatism $1 - \alpha$ goes to 0 (this dependence is studied by Member 3).

**Loop vs matrix version.** The two versions produce the same iterates up to round-off ($2.2 \times 10^{-16}$), as they must, since they compute the same formula. The pure Python loop takes about 99 ms per iteration against 0.54 ms for the matrix version, about 180 times slower. Both perform the same $\approx 5 \times 10^5$ multiply-adds per iteration, but in the loop each of them goes through the Python interpreter, while the sparse product `R @ x` runs a compiled C loop over the same CSR arrays. Compiling our own loop with `numba.njit` brings it to 0.82 ms per sweep, the same order as scipy: the algorithm is not slow, the interpreter is. The matrix version is the default (`mode="matrix"`). Its only extra cost is storing the off-diagonal part $A - D$ once (about 7 MB).

**Cost.** One Jacobi iteration costs one sparse matrix-vector product, $O(\mathrm{nnz}(A))$ operations, and needs no memory beyond a few vectors of length $n$. The whole solve takes about 0.06 s, against about 150 s and 1.7 GB for the sparse LU of Section 1.

## 3. Gauss-Seidel method

**Goal.** Implement Gauss-Seidel from scratch (loop and matrix versions), validate it as we did for Jacobi, compare it with Jacobi, and check how much it depends on the ordering of the nodes (original, RCM, random, reversed).

**Method.** Split $A = L + D + U$ (strictly lower, diagonal, strictly upper part). Gauss-Seidel keeps on the left the lower triangle, which uses the components already updated in the current sweep:

$$(D + L)\,x^{(k+1)} = b - U x^{(k)}, \qquad B_{GS} = -(D + L)^{-1} U.$$

For our system $D + L = I - \alpha W_L$ and $-U = \alpha W_U$, where $W_L$ and $W_U$ are the strictly lower and upper parts of $W$.

- **Loop version** (`mode="loop"`): the same row formula as Jacobi, but $x$ is overwritten in place. When row $i$ is processed, $x_j$ already holds the new value for $j < i$ and the old one for $j > i$.
- **Matrix version** (`mode="matrix"`): one product $U x^{(k)}$ and one sparse lower triangular solve by forward substitution (`scipy.sparse.linalg.spsolve_triangular`). No factorization is computed, so there is no fill-in.

Since $A$ is strictly diagonally dominant by rows, Gauss-Seidel converges for any $x^{(0)}$. Unlike Jacobi, the result of a sweep depends on which unknowns come first, so the ordering of the nodes matters. To solve the permuted system we use $P A P^T (P x) = P b$, i.e. `A[p, :][:, p]` and `b[p]`, then undo the permutation with `x[p] = x_perm`.

In [13]:
# 3.1 Validation on the same small system (b = A @ ones)

from solvers import gauss_seidel, _gauss_seidel_sweep

for mode in ["matrix", "loop"]:
    x_small, hist_small = gauss_seidel(A_small, b_small, tol=1e-12, mode=mode)
    print(f"{mode:6s}: {hist_small.attrs['reason']}, max |x - 1| = {np.abs(x_small - 1).max():.1e}")

matrix: residual <= 1e-12 at iteration 14, max |x - 1| = 1.4e-12
loop  : residual <= 1e-12 at iteration 14, max |x - 1| = 1.4e-12


In [14]:
# 3.2 Full system, matrix version, compared with x* and with Jacobi

x_gs, hist_gs = gauss_seidel(A, b, tol=1e-8, criterion="residual", mode="matrix")

rel_err_gs = np.linalg.norm(x_gs - x_ref, np.inf) / np.linalg.norm(x_ref, np.inf)
print(hist_gs.attrs)
print(f"relative error ||x - x*||_inf / ||x*||_inf = {rel_err_gs:.2e}")
print(f"iterations: Gauss-Seidel {len(hist_gs)}, Jacobi {len(hist_jac)}")
print(f"time per iteration: Gauss-Seidel {hist_gs.time_s.mean() * 1e3:.2f} ms, Jacobi {hist_jac.time_s.mean() * 1e3:.2f} ms")
print(f"total time: Gauss-Seidel {hist_gs.time_s.sum():.3f} s, Jacobi {hist_jac.time_s.sum():.3f} s")

ratio_gs = hist_gs.rel_residual.to_numpy()[1:] / hist_gs.rel_residual.to_numpy()[:-1]
print("residual ratio, first 3 iterations:", np.round(ratio_gs[:3], 4), f"(alpha^2 = {alpha**2:.4f})")
print("residual ratio, last 3 iterations: ", np.round(ratio_gs[-3:], 4))

{'converged': True, 'reason': 'residual <= 1e-08 at iteration 68'}
relative error ||x - x*||_inf / ||x*||_inf = 5.78e-09
iterations: Gauss-Seidel 68, Jacobi 114
time per iteration: Gauss-Seidel 7.41 ms, Jacobi 0.47 ms
total time: Gauss-Seidel 0.504 s, Jacobi 0.054 s
residual ratio, first 3 iterations: [0.7225 0.7225 0.7225] (alpha^2 = 0.7225)
residual ratio, last 3 iterations:  [0.7635 0.7635 0.7635]


In [15]:
# 3.3 Loop version: same iterates as the matrix version
# The pure Python loop is slow (about 0.1 s per sweep), so we compare 5 iterations only.

x_gs_loop, hist_gs_loop = gauss_seidel(A, b, max_iter=5, mode="loop")
x_gs_mat5, _ = gauss_seidel(A, b, max_iter=5, mode="matrix")
print(f"after 5 iterations: max |x_loop - x_matrix| = {np.abs(x_gs_loop - x_gs_mat5).max():.1e}")

# numba version of the in-place sweep
gs_sweep_numba = njit(_gauss_seidel_sweep)
gs_sweep_numba(A.indptr, A.indices, A.data, diag, b, np.zeros(n))  # first call compiles

start = time.perf_counter()
x_tmp = np.zeros(n)
for _ in range(n_rep):
    gs_sweep_numba(A.indptr, A.indices, A.data, diag, b, x_tmp)
gs_numba_time = (time.perf_counter() - start) / n_rep

print(f"one Gauss-Seidel sweep: python loop {hist_gs_loop.time_s.mean() * 1e3:.1f} ms, "
      f"matrix (spsolve_triangular) {hist_gs.time_s.mean() * 1e3:.2f} ms, numba {gs_numba_time * 1e3:.2f} ms")
print(f"after {n_rep} numba sweeps: relative error {np.linalg.norm(x_tmp - x_ref, np.inf) / np.linalg.norm(x_ref, np.inf):.1e}")

after 5 iterations: max |x_loop - x_matrix| = 2.2e-16
one Gauss-Seidel sweep: python loop 101.6 ms, matrix (spsolve_triangular) 7.41 ms, numba 0.85 ms
after 100 numba sweeps: relative error 1.0e-12


In [16]:
# 3.4 Effect of the node ordering
# Jacobi does not depend on the ordering, Gauss-Seidel does.

import pandas as pd

orderings = {
    "original": np.arange(n),
    "RCM": p,
    "random (seed 42)": np.random.default_rng(42).permutation(n),
    "reversed": np.arange(n)[::-1],
}

rows = []
for name, perm in orderings.items():
    A_perm = A[perm, :][:, perm]
    b_perm = b[perm]
    x_perm, h_gs = gauss_seidel(A_perm, b_perm, tol=1e-8)
    _, h_jac = jacobi(A_perm, b_perm, tol=1e-8)

    x_back = np.empty(n)
    x_back[perm] = x_perm  # undo the permutation
    W_perm = W[perm, :][:, perm]
    rows.append({"ordering": name,
                 "GS iterations": len(h_gs),
                 "Jacobi iterations": len(h_jac),
                 "GS rel. error": np.linalg.norm(x_back - x_ref, np.inf) / np.linalg.norm(x_ref, np.inf),
                 "nnz(W_L)": sp.tril(W_perm, k=-1).nnz,
                 "nnz(W_U)": sp.triu(W_perm, k=1).nnz})

ordering_table = pd.DataFrame(rows)
ordering_table

,ordering,GS iterations,Jacobi iterations,GS rel. error,nnz(W_L),nnz(W_U)
0,original,68,114,5.782931e-09,287661,221176
1,RCM,66,114,4.969751e-09,222927,285910
2,random (seed 42),68,114,6.438503e-09,253076,255761
3,reversed,70,114,5.176688e-09,221176,287661


### Results of Section 3

**Validation.** On the small test matrix both versions reach $x = (1, \dots, 1)$ with error $1.4 \times 10^{-12}$ in 14 iterations, against 26 for Jacobi. On the full system Gauss-Seidel stops after 68 iterations with relative error $5.8 \times 10^{-9}$. The loop and matrix versions give the same iterates up to round-off ($2.2 \times 10^{-16}$).

**Gauss-Seidel needs fewer iterations: 68 against 114.** Using the new components as soon as they are available makes the error decrease faster. In the first iterations the residual is reduced by exactly $0.7225 = \alpha^2$ per sweep, i.e. one Gauss-Seidel sweep is worth two Jacobi sweeps. Asymptotically the factor settles at about $0.76$: still clearly better than Jacobi ($0.85$), but not the ideal $\rho(B_{GS}) = \rho(B_J)^2$ that holds for some special classes of matrices (e.g. tridiagonal ones). Hence $68 / 114 \approx 0.6$ and not $0.5$. The exact value of $\rho(B_{GS})$ is computed by Member 3.

**Fewer iterations does not automatically mean less time.** The same work per sweep ($\approx \mathrm{nnz}(A)$ multiply-adds) takes:

| one sweep | Jacobi | Gauss-Seidel |
|---|---|---|
| pure Python loop | 99 ms | 102 ms |
| matrix version (scipy) | 0.5 ms (`R @ x`) | 7.4 ms (`spsolve_triangular`) |
| loop compiled with numba | 0.8 ms | 0.85 ms |

In loop form the two methods cost the same, as expected. In matrix form Gauss-Seidel requires a triangular solve by forward substitution, which is intrinsically sequential (row $i$ needs $x_1, \dots, x_{i-1}$ of the same sweep), and the scipy routine is about 15 times slower than a sparse matrix-vector product. With the scipy matrix versions Jacobi is therefore faster overall (0.05 s against 0.5 s). With compiled loops the two totals are about the same ($114 \times 0.8$ ms against $68 \times 0.85$ ms, both about 0.06 s). Jacobi has another practical advantage: its rows are independent, so it parallelizes trivially, while Gauss-Seidel is sequential by construction.

**Ordering.** Jacobi needs 114 iterations with every ordering, as expected: permuting rows and columns together gives $P B_J P^T$, which has the same eigenvalues. Gauss-Seidel changes, but only slightly: 66 iterations with RCM, 68 with the original and a random ordering, 70 with the reversed one. The final error is around $5 \times 10^{-9}$ in all cases. The number of entries that fall in the lower triangle ($W_L$, the part that uses new values) does not explain the differences: RCM has fewer of them than the original ordering and still converges slightly faster. RCM is designed to reduce the bandwidth for direct methods (Section 5), not to accelerate Gauss-Seidel, and here its effect on Gauss-Seidel is of the order of 3%. A plausible (not verified) explanation is that the slowest error components live in the small closed groups of users found in Section 2, whose internal ordering a global reordering barely changes.